# components/selection_panel

> Selection panel component showing selected files with drag-drop reordering

In [ ]:
#| default_exp components.selection_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List
import json

from fasthtml.common import Div, Span, Button, Form, Ul, Li, P, Hidden

from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_sizes, badge_styles
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui
from cjm_fasthtml_daisyui.utilities.border_radius import border_radius

from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, min_h
from cjm_fasthtml_tailwind.utilities.typography import (
    font_size, font_weight, text_align, truncate, list_style
)
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import border
from cjm_fasthtml_tailwind.utilities.effects import shadow
from cjm_fasthtml_tailwind.utilities.interactivity import cursor
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, grow, shrink, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds
from cjm_transcription_source_select.utils import format_file_size, format_duration

## Type Badge

In [ ]:
#| export
def _render_type_badge(
    file_type: str,  # "audio" or "video"
) -> Span:  # Badge component
    """Render a type badge for a selected file."""
    if file_type == "video":
        return Span("video", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.secondary))
    return Span("audio", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.info))

## Queue Item

In [ ]:
#| export
def render_queue_item(
    selected_file: SelectedFile,  # Selected file data
    index: int,  # Position in queue (1-based)
    urls: SourceSelectUrls,  # URL bundle
) -> Li:  # Queue item element
    """Render a single item in the selection queue."""
    path = selected_file["path"]
    filename = selected_file.get("filename", path.rsplit("/", 1)[-1])
    file_type = selected_file.get("file_type", "audio")
    size_bytes = selected_file.get("size_bytes", 0)

    return Li(
        Hidden(name="item", value=path),

        # Drag handle
        Span(
            lucide_icon("grip-vertical", size=4, cls=str(text_dui.base_content.opacity(40))),
            cls=combine_classes("drag-handle", cursor.move, p.r(2))
        ),

        # Position number
        Span(f"{index}.", cls=combine_classes(font_weight.bold, p.r(2), w(6))),

        # Filename
        Span(
            filename,
            cls=combine_classes(grow(), font_size.sm, truncate),
            title=path
        ),

        # Type badge
        _render_type_badge(file_type),

        # Size
        Span(
            format_file_size(size_bytes) if size_bytes else "",
            cls=combine_classes(font_size.xs, text_dui.base_content.opacity(60), m.l(2))
        ),

        # Preview button
        Button(
            lucide_icon("eye", size=4, cls=str(text_dui.base_content.opacity(60))),
            cls=combine_classes(btn, btn_styles.ghost, btn_sizes.xs, m.l(1)),
            hx_post=urls.preview,
            hx_vals=json.dumps({"path": path}),
            hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.PREVIEW_PANEL),
            hx_swap="outerHTML",
            title="Preview file"
        ),

        # Remove button
        Button(
            lucide_icon("x", size=4, cls=str(text_dui.base_content.opacity(60))),
            cls=combine_classes(btn, btn_styles.ghost, btn_sizes.xs, m.l(1)),
            hx_post=urls.remove,
            hx_vals=json.dumps({"path": path}),
            hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.SELECTION_PANEL),
            hx_swap="outerHTML",
            title="Remove from selection"
        ),

        id=SourceSelectHtmlIds.selection_item(path),
        cls=combine_classes(
            "queue-item",
            str(flex_display), items.center,
            m.x(0), m.y(1), p(3),
            bg_dui.base_100, bg_dui.base_200.hover
        ),
        data_path=path
    )

## Selection Panel

In [ ]:
#| export
def render_selection_panel(
    selected_files: List[SelectedFile],  # Ordered list of selected files
    urls: SourceSelectUrls,  # URL bundle
) -> Div:  # Selection panel component
    """Render the selection panel with drag-drop reordering."""
    count = len(selected_files)

    # Queue items
    queue_items = [
        render_queue_item(f, i + 1, urls)
        for i, f in enumerate(selected_files)
    ]

    # Empty state
    empty_state = Div(
        P("No files selected",
          cls=combine_classes(text_dui.base_content.opacity(40), text_align.center, font_size.sm)),
        P("Click files in the browser to select them",
          cls=combine_classes(text_dui.base_content.opacity(30), text_align.center, font_size.xs)),
        id=SourceSelectHtmlIds.SELECTION_EMPTY,
        cls=combine_classes(p(8), str(flex_display), flex_direction.col, justify.center, items.center, grow())
    ) if not selected_files else None

    # Header
    header = Div(
        Div(
            Span("Selected", cls=str(font_weight.bold)),
            Span(
                str(count),
                cls=combine_classes(badge, badge_colors.primary, badge_sizes.sm, m.l(2))
            ),
            cls=str(flex_display)
        ),
        Button(
            "Clear",
            cls=combine_classes(btn, btn_styles.ghost, btn_sizes.xs),
            hx_post=urls.clear,
            hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.SELECTION_PANEL),
            hx_swap="outerHTML"
        ) if selected_files else None,
        cls=combine_classes(
            str(flex_display), justify.between, items.center,
            p(3), border_dui.base_200, border.b()
        )
    )

    # Queue list with SortableJS
    queue_list = Ul(
        *queue_items,
        id=SourceSelectHtmlIds.SELECTION_LIST,
        hx_post=urls.reorder,
        hx_trigger="end",
        hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.SELECTION_PANEL),
        hx_swap="outerHTML",
        cls=combine_classes("sortable", grow(), overflow.y.auto, list_style.none, m(0), p(0))
    ) if selected_files else empty_state

    return Div(
        header,
        queue_list,
        id=SourceSelectHtmlIds.SELECTION_PANEL,
        cls=combine_classes(
            w.full, min_h(48),
            bg_dui.base_100, border_radius.box,
            shadow.lg, border_dui.base_300,
            str(flex_display), flex_direction.col,
            overflow.hidden
        )
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()